# 04 多因子信号合成与策略对比

展示：
1. 多因子 IC 加权合成信号
2. 信号可视化（权重演化 + 信号值）
3. 五个策略横向对比

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from futuresquant.data.loader import FuturesDataLoader
from futuresquant.backtest.engine import BacktestEngine, BacktestConfig
from futuresquant.backtest.analyzer import PerformanceAnalyzer
from futuresquant.factors.signal import MultiFactorSignal
from futuresquant.factors.technical import ROC, MACross, NormATR, RSI, TSMomentum, OBV
from futuresquant.strategy.examples import (
    MACrossStrategy, ATRBreakoutStrategy,
    RSIMeanReversionStrategy, BollingerBandStrategy,
    MultiFactorStrategy,
)

DATA_DIR = r'I:\stock\FuturesQuant\raw_data\1min_FU'
CACHE_DIR = r'I:\stock\FuturesQuant\cache'
SYMBOL = 'FU2210'
CONFIG = BacktestConfig(symbol=SYMBOL, initial_capital=1_000_000)

loader = FuturesDataLoader(DATA_DIR, cache_dir=CACHE_DIR)
klines = loader.load(SYMBOL, start='2022-01-01', end='2022-10-31')
print(f'K线行数: {len(klines):,}')

K线行数: 23,982


## 1. 构建多因子信号

In [2]:
signal_gen = MultiFactorSignal(
    factors=[ROC(20), MACross(5, 20), NormATR(14), RSI(14), TSMomentum(5, 20), OBV(20)],
    forward_bars=60,
    ic_window=240 * 5,    # 5天
    norm_window=240 * 20, # 20天
    min_ic_abs=0.005,
)

signal = signal_gen.compute(klines)
weights = signal_gen.factor_weights(klines)

print(f'信号非空数: {signal.notna().sum():,}')
print(f'信号范围: [{signal.min():.2f}, {signal.max():.2f}]')
print(f'信号均值: {signal.mean():.4f}   标准差: {signal.std():.4f}')

信号非空数: 14,361
信号范围: [-5.21, 7.93]
信号均值: 0.0612   标准差: 1.0690


## 2. 信号与价格叠加可视化

In [3]:
# 日线降采样便于可视化
daily_signal = signal.resample('1D').last().dropna()
daily_close = klines['close'].resample('1D').last().dropna()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['标的价格', '多因子合成信号'],
                    row_heights=[0.6, 0.4], vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=daily_close.index, y=daily_close.values,
                          name='FU2210', line=dict(color='#1f77b4')), row=1, col=1)

# 信号着色：正 = 绿（偏多），负 = 红（偏空）
colors = ['#2ca02c' if v > 0 else '#d62728' for v in daily_signal.values]
fig.add_trace(go.Bar(x=daily_signal.index, y=daily_signal.values,
                      name='Signal', marker_color=colors, opacity=0.7), row=2, col=1)
fig.add_hline(y=1.0, line_dash='dash', line_color='green',
               annotation_text='做多阈值', row=2, col=1)
fig.add_hline(y=-1.0, line_dash='dash', line_color='red',
               annotation_text='做空阈值', row=2, col=1)
fig.add_hline(y=0, line_color='black', line_width=0.5, row=2, col=1)

fig.update_layout(title='多因子合成信号 vs 价格', height=550, hovermode='x unified')
fig.show()

## 3. 因子权重演化

In [4]:
daily_weights = weights.resample('1D').last().dropna(how='all')

fig = go.Figure()
colors_list = px.colors.qualitative.Plotly
for i, col in enumerate(daily_weights.columns):
    fig.add_trace(go.Scatter(
        x=daily_weights.index, y=daily_weights[col],
        name=col, mode='lines',
        line=dict(color=colors_list[i % len(colors_list)], width=1.5)
    ))

fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.update_layout(
    title='因子动态权重演化（IC 自适应加权）',
    yaxis_title='权重', height=400,
    hovermode='x unified'
)
fig.show()

## 4. 各策略回测并行运行

In [5]:
strategies = {
    'MA双均线(5/20)':       MACrossStrategy(5, 20),
    'ATR通道突破(20)':      ATRBreakoutStrategy(20, 14),
    'RSI均值回归(14)':      RSIMeanReversionStrategy(14, oversold=30, overbought=70),
    '布林带回归(20)':       BollingerBandStrategy(20, mode='reversion'),
    '布林带突破(20)':       BollingerBandStrategy(20, mode='breakout'),
    '多因子信号':           MultiFactorStrategy(signal_gen, entry_threshold=1.0),
}

results = {}
for name, strat in strategies.items():
    print(f'运行: {name}...')
    results[name] = BacktestEngine(strat, CONFIG).run(klines)

print('完成！')

Output()

运行: MA双均线(5/20)...


Output()

运行: ATR通道突破(20)...


Output()

...

Output()

运行: 布林带突破(20)...


运行: 多因子信号...


Output()

完成！


## 5. 权益曲线对比

In [6]:
fig = go.Figure()
for name, result in results.items():
    eq = result.account.equity_curve().resample('1D').last().dropna()
    # 归一化为初始资金 = 1
    eq_norm = eq / CONFIG.initial_capital
    fig.add_trace(go.Scatter(x=eq_norm.index, y=eq_norm.values, name=name, mode='lines'))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray', line_width=0.8)
fig.update_layout(
    title='各策略归一化权益曲线对比',
    yaxis_title='归一化权益', height=450,
    hovermode='x unified'
)
fig.show()

## 6. 绩效指标对比表

In [7]:
records = []
for name, result in results.items():
    m = result.metrics
    records.append({
        '策略':       name,
        '总收益':     f"{m['total_return']:.2%}",
        '年化收益':   f"{m['annual_return']:.2%}",
        '年化波动':   f"{m['annual_vol']:.2%}",
        'Sharpe':    f"{m['sharpe']:.3f}",
        'Sortino':   f"{m['sortino']:.3f}",
        '最大回撤':   f"{m['max_drawdown']:.2%}",
        'Calmar':    f"{m['calmar']:.3f}",
        '成交笔数':   len(result.account.fills),
    })

compare_df = pd.DataFrame(records).set_index('策略')
compare_df

,总收益,年化收益,年化波动,Sharpe,Sortino,最大回撤,Calmar,成交笔数
策略,,,,,,,,
MA双均线(5/20),2.46%,6.27%,5.27%,1.190,1.031,-2.38%,2.633,3002
ATR通道突破(20),0.00%,0.00%,0.00%,0.000,0.000,0.00%,0.000,0
RSI均值回归(14),0.00%,0.00%,0.00%,0.000,0.000,0.00%,0.000,0
布林带回归(20),0.05%,0.13%,21.15%,0.006,0.005,-1.65%,0.078,956
布林带突破(20),1.78%,4.52%,21.61%,0.209,0.166,-1.52%,2.977,956
多因子信号,-1.65%,-4.08%,17.96%,-0.227,-0.119,-2.45%,-1.663,748


## 7. Sharpe / 最大回撤 散点图

In [8]:
plot_data = pd.DataFrame([
    {
        '策略': name,
        'Sharpe': result.metrics['sharpe'],
        'MaxDD': result.metrics['max_drawdown'] * 100,
        '年化收益': result.metrics['annual_return'] * 100,
        '成交笔数': len(result.account.fills),
    }
    for name, result in results.items()
])

fig = px.scatter(
    plot_data, x='MaxDD', y='Sharpe',
    size='成交笔数', color='策略', text='策略',
    title='策略风险收益分布（气泡大小 = 成交笔数）',
    labels={'MaxDD': '最大回撤 (%)', 'Sharpe': 'Sharpe 比率'},
    size_max=40,
)
fig.update_traces(textposition='top center')
fig.add_hline(y=0, line_color='gray', line_dash='dash')
fig.update_layout(height=450, showlegend=False)
fig.show()

## 8. 多因子策略：信号强度与持仓关系

In [9]:
mf_result = results['多因子信号']
fills_df = pd.DataFrame([
    {'time': f.time, 'direction': f.direction.value,
     'offset': f.offset.value, 'price': f.price}
    for f in mf_result.account.fills
])

daily_close = klines['close'].resample('1D').last().dropna()
daily_sig = signal.resample('1D').last().dropna()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['价格 + 成交点', '多因子信号'],
                    row_heights=[0.6, 0.4], vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=daily_close.index, y=daily_close.values,
                          name='价格', line=dict(color='gray', width=1)), row=1, col=1)

if not fills_df.empty:
    buy_opens = fills_df[(fills_df['direction']=='LONG') & (fills_df['offset']=='OPEN')]
    sell_opens = fills_df[(fills_df['direction']=='SHORT') & (fills_df['offset']=='OPEN')]
    fig.add_trace(go.Scatter(
        x=buy_opens['time'], y=buy_opens['price'],
        mode='markers', name='买入', marker=dict(symbol='triangle-up', size=9, color='#2ca02c')
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=sell_opens['time'], y=sell_opens['price'],
        mode='markers', name='卖出', marker=dict(symbol='triangle-down', size=9, color='#d62728')
    ), row=1, col=1)

colors = ['#2ca02c' if v > 0 else '#d62728' for v in daily_sig.values]
fig.add_trace(go.Bar(x=daily_sig.index, y=daily_sig.values,
                      marker_color=colors, opacity=0.6, name='信号'), row=2, col=1)
for thresh in [1.0, -1.0]:
    fig.add_hline(y=thresh, line_dash='dash',
                   line_color='green' if thresh > 0 else 'red', row=2, col=1)

fig.update_layout(title='多因子策略：信号驱动的交易点', height=550, hovermode='x unified')
fig.show()

print(f'总开仓次数: {len(fills_df[fills_df["offset"]=="OPEN"]) if not fills_df.empty else 0}')

总开仓次数: 374
